# Task 1: Descriptive Analysis

Goal: 
1. Loads the analytic dataset
2. Reports missingness for all variables
3. Generates Table 1 (baseline characteristics by AF status)

In [ ]:
# Installation

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Data

In [ ]:
# Data

df = pd.read_csv('data/data.csv')

print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')

## 2. Basic Overview

In [ ]:
n = len(df)
n_af    = df['has_af_index'].sum()
n_noaf  = n - n_af
n_died  = df['hospital_expire_flag'].sum()

print('=' * 50)
print('FINAL COHORT OVERVIEW')
print('=' * 50)

print(f'Total patients: {n:,}')
print(f'With AF: {n_af:,} ({100*n_af/n:.1f}%)')
print(f'Without AF: {n_noaf:,} ({100*n_noaf/n:.1f}%)')
print(f'In-hospital deaths: {n_died:,} ({100*n_died/n:.1f}%)')
print(f'\nHF Subtypes:')
print(f'HFrEF: {df.is_hfref.sum():,} ({100*df.is_hfref.mean():.1f}%)')
print(f'HFpEF: {df.is_hfpef.sum():,} ({100*df.is_hfpef.mean():.1f}%)')
print(f'Mixed: {df.is_hfmixed.sum():,} ({100*df.is_hfmixed.mean():.1f}%)')

FINAL COHORT OVERVIEW
Total patients: 9,463
With AF: 3,776 (39.9%)
Without AF: 5,687 (60.1%)
In-hospital deaths: 1,265 (13.4%)

HF Subtypes:
HFrEF: 4,577 (48.4%)
HFpEF: 4,245 (44.9%)
Mixed: 685 (7.2%)


## 3. Cohort Flow Diagram

In [10]:
flow = pd.DataFrame({
    'Step': [
        '1. Acute HF admissions (Yan 2023 ICD codes)',
        '2. Joined with demographics',
        '3. Age ≥ 18, ICU ≥ 24h, not AMA/hospice, has dischtime',
        '4. First qualifying admission per patient',
        '5. No prior AF — FINAL COHORT'
    ],
    'N Admissions': [66008, 66008, 15553, 11894, 9463],
    'N Patients':   [26391, 26391, 11894, 11894, 9463]
})

print(flow.to_string(index=False))

                                                  Step  N Admissions  N Patients
           1. Acute HF admissions (Yan 2023 ICD codes)         66008       26391
                           2. Joined with demographics         66008       26391
3. Age ≥ 18, ICU ≥ 24h, not AMA/hospice, has dischtime         15553       11894
             4. First qualifying admission per patient         11894       11894
                         5. No prior AF — FINAL COHORT          9463        9463


## 4. Missingness Analysis

In [11]:
miss = pd.DataFrame({
    'Variable':    df.columns,
    'N Missing':   df.isnull().sum().values,
    'Pct Missing': (df.isnull().mean() * 100).round(1).values
}).sort_values('Pct Missing', ascending=False)

print('MISSINGNESS REPORT')
print(miss[miss['N Missing'] > 0].to_string(index=False))

high = miss[miss['Pct Missing'] > 20]
if len(high):
    print(f'Missingness >20%: {list(high.Variable)}')
else:
    print('There is no variables that has >20% missingness')

MISSINGNESS REPORT
          Variable  N Missing  Pct Missing
         deathtime       8199        86.60
        temp_max_c        482         5.10
           wbc_max        119         1.30
           rdw_max        125         1.30
    hemoglobin_min        112         1.20
     platelets_min        112         1.20
         insurance         76         0.80
     potassium_max         72         0.80
     anion_gap_max         79         0.80
           sbp_min         68         0.70
           dbp_min         68         0.70
       glucose_max         65         0.70
        sodium_min         66         0.70
           bun_max         67         0.70
    creatinine_max         65         0.70
           mbp_min         67         0.70
            rr_max         18         0.20
          spo2_min         15         0.20
    heart_rate_max         13         0.10
discharge_location         13         0.10
Missingness >20%: ['deathtime']


## 5. Table 1 — Baseline Characteristics by AF Status

In [ ]:
# Charactertistics

def recode_race(r):
    if pd.isna(r): return 'Other/Unknown'
    r = str(r).upper()
    if 'WHITE' in r: return 'White'
    if 'BLACK' in r: return 'Black'
    if 'ASIAN' in r: return 'Asian'
    if 'HISPANIC' in r or 'LATINO' in r: return 'Hispanic'
    return 'Other/Unknown'

df['race_group'] = df['race'].apply(recode_race)

af0 = df[df['has_af_index'] == 0]
af1 = df[df['has_af_index'] == 1]

def fmt_p(p):
    if pd.isna(p): return ''
    if p < 0.01: return '<0.01'
    return f'{p:.3f}'

def mean_sd(series):
    s = series.dropna()
    return f"{s.mean():.1f} ± {s.std():.1f}"

def med_iqr(series):
    s = series.dropna()
    return f"{s.median():.1f} [{s.quantile(0.25):.1f}–{s.quantile(0.75):.1f}]"

def pct(series, denom):
    n = int(series.sum())
    return f"{n} ({100*n/denom:.1f}%)"

def t_p(col, g0, g1):
    a, b = g0[col].dropna(), g1[col].dropna()
    if len(a) < 5 or len(b) < 5: return np.nan
    _, p = stats.ttest_ind(a, b, equal_var=False)
    return fmt_p(p)

def mwu_p(col, g0, g1):
    a, b = g0[col].dropna(), g1[col].dropna()
    if len(a) < 5 or len(b) < 5: return np.nan
    _, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    return fmt_p(p)

def chi2_p(col, df):
    ct = pd.crosstab(df[col], df['has_af_index'])
    if ct.shape[0] < 2: return np.nan
    _, p, _, _ = stats.chi2_contingency(ct)
    return fmt_p(p)

rows = []
def add(var, overall, no_af, af, p=''):
    rows.append({'Variable': var, f'Overall (n={len(df)})': overall,
                 f'No AF (n={len(af0)})': no_af,
                 f'AF (n={len(af1)})': af, 'p-value': p})

# Demographics
add('DEMOGRAPHICS', '', '', '')
add('Age, mean ± SD', mean_sd(df.age_at_admit), mean_sd(af0.age_at_admit), mean_sd(af1.age_at_admit), t_p('age_at_admit', af0, af1))
add('Female, n (%)', pct(df.gender=='F', len(df)), pct(af0.gender=='F', len(af0)), pct(af1.gender=='F', len(af1)), chi2_p('gender', df))

add('Race, n (%)', '', '', '', chi2_p('race_group', df))
for r in ['White','Black','Hispanic','Asian','Other/Unknown']:
    add(f'  {r}', pct(df.race_group==r, len(df)), pct(af0.race_group==r, len(af0)), pct(af1.race_group==r, len(af1)))

add('Insurance, n (%)', '', '', '', chi2_p('insurance', df))
for ins in sorted(df['insurance'].dropna().unique()):
    add(f'  {ins}', pct(df.insurance==ins, len(df)), pct(af0.insurance==ins, len(af0)), pct(af1.insurance==ins, len(af1)))

# HF Subtype
add('HF SUBTYPE', '', '', '')
add('HFrEF, n (%)', pct(df.is_hfref, len(df)), pct(af0.is_hfref, len(af0)), pct(af1.is_hfref, len(af1)), chi2_p('is_hfref', df))
add('HFpEF, n (%)', pct(df.is_hfpef, len(df)), pct(af0.is_hfpef, len(af0)), pct(af1.is_hfpef, len(af1)), chi2_p('is_hfpef', df))
add('Mixed HF, n (%)', pct(df.is_hfmixed, len(df)), pct(af0.is_hfmixed, len(af0)), pct(af1.is_hfmixed, len(af1)), chi2_p('is_hfmixed', df))

# ICU Unit
add('ICU UNIT', '', '', '', chi2_p('careunit_cat', df))
for u in ['Cardiac','Medical','Surgical/Other']:
    add(f'  {u}', pct(df.careunit_cat==u, len(df)), pct(af0.careunit_cat==u, len(af0)), pct(af1.careunit_cat==u, len(af1)))

# Comorbidities
add('COMORBIDITIES', '', '', '')
for col, label in [('has_htn','Hypertension'),('has_dm','Diabetes'),
                   ('has_ckd','CKD'),('has_cad','CAD'),
                   ('has_copd','COPD'),('has_valvular','Valvular disease'),
                   ('has_pad','PAD'),('has_stroke_hx','Stroke history'),
                   ('has_pulm_htn','Pulmonary HTN'),('has_cardiomyopathy','Cardiomyopathy')]:
    add(f'{label}, n (%)', pct(df[col], len(df)), pct(af0[col], len(af0)), pct(af1[col], len(af1)), chi2_p(col, df))

# Labs (first 24h) — median [IQR] because min/max values are skewed across patients
add('LABS (first 24h)', '', '', '')
for col, label in [('wbc_max','WBC max (K/uL)'),('hemoglobin_min','Hemoglobin min (g/dL)'),
                   ('platelets_min','Platelets min (K/uL)'),('creatinine_max','Creatinine max (mg/dL)'),
                   ('bun_max','BUN max (mg/dL)'),('sodium_min','Sodium min (mEq/L)'),
                   ('potassium_max','Potassium max (mEq/L)'),('glucose_max','Glucose max (mg/dL)'),
                   ('anion_gap_max','Anion gap max'),('rdw_max','RDW max (%)')]:
    add(f'{label}, median [IQR]', med_iqr(df[col]), med_iqr(af0[col]), med_iqr(af1[col]), mwu_p(col, af0, af1))

# Vitals (first 24h) — median [IQR]
add('VITALS (first 24h)', '', '', '')
for col, label in [('heart_rate_max','Heart rate max (bpm)'),('sbp_min','SBP min (mmHg)'),
                   ('dbp_min','DBP min (mmHg)'),('mbp_min','MBP min (mmHg)'),
                   ('rr_max','RR max (breaths/min)'),('spo2_min','SpO2 min (%)'),
                   ('temp_max_c','Temp max (°C)')]:
    add(f'{label}, median [IQR]', med_iqr(df[col]), med_iqr(af0[col]), med_iqr(af1[col]), mwu_p(col, af0, af1))

# Outcomes
add('OUTCOMES', '', '', '')
add('In-hospital death, n (%)', pct(df.hospital_expire_flag, len(df)), pct(af0.hospital_expire_flag, len(af0)), pct(af1.hospital_expire_flag, len(af1)), chi2_p('hospital_expire_flag', df))
add('30-day all-cause death, n (%)', pct(df.death_30d, len(df)), pct(af0.death_30d, len(af0)), pct(af1.death_30d, len(af1)), chi2_p('death_30d', df))
add('Time to event 30d, median [IQR] (days)', med_iqr(df.time_to_event_30d), med_iqr(af0.time_to_event_30d), med_iqr(af1.time_to_event_30d), mwu_p('time_to_event_30d', af0, af1))
add('ICU LOS, median [IQR] (days)', med_iqr(df.icu_total_los), med_iqr(af0.icu_total_los), med_iqr(af1.icu_total_los), mwu_p('icu_total_los', af0, af1))

table1 = pd.DataFrame(rows)
table1.to_csv('characteristics.csv', index=False)
table1

## 6. Distribution Plots by AF Status

In [ ]:
plot_vars = [
    ('age_at_admit', 'Age at admission'),
    ('icu_total_los', 'ICU LOS (days)'),
    ('creatinine_max', 'Creatinine max'),
    ('hemoglobin_min', 'Hemoglobin min'),
    ('heart_rate_max', 'Heart rate max'),
    ('sbp_min', 'SBP min'),
    ('spo2_min', 'SpO2 min'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (var, label) in enumerate(plot_vars):
    ax = axes[i]
    for grp, lbl, col in [(0, 'No AF', 'blue'), (1, 'AF', 'red')]:
        data = df[df['has_af_index'] == grp][var].dropna()
        p1, p99 = data.quantile(0.01), data.quantile(0.99)
        data = data.clip(p1, p99)
        ax.hist(data, bins = 30, alpha = 0.6, color = col, label = lbl, density = True)
    ax.set_title(label, fontsize = 10, fontweight='bold')
    ax.legend(fontsize = 8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# hide the unused 8th subplot
axes[-1].set_visible(False)

plt.suptitle('Distribution of Key Variables by AF Status', fontsize = 10, y = 1)
plt.tight_layout()
plt.savefig('distributions.png')
plt.show()

# Conclusions
- 9,463 patients in final cohort
- 39.9% with hospital-documented AF
- 13.4% in-hospital mortality
- Missingness very low (<6% for all variables except deathtime)